# Source reconstruction

While MEG data is collected using sensors outside the skull, it can be used to estimate neural sources in the brain. This requires a **forward model**, which predicts how neural activity generates MEG signals, and an **inverse model**, which uses this prediction to estimate the sources responsible for the recorded data.


In [ ]:
# we need the newest version of mne otherwise you'll get an error
!pip install --upgrade mne

In [ ]:
import mne
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# importing from our local utils
import sys
sys.path.append("../scripts/")
from params import fnames, L_FREQ, H_FREQ, FILT_METHOD

## Preparing the data

In [ ]:
# Preparing paths
# TODO: Update so it matches the path to the data from your specific participant
subject = "0193"

epochs_path = fnames.sub_filtered_ica_epochs(subject=subject, l_freq = L_FREQ, h_freq=H_FREQ, method=FILT_METHOD) # or which ever way you defined it in your group!
trans_path = fnames.head_mri_t(subject=subject)
behav_path = fnames.sub_behav(subject=subject)
subjects_dir = fnames.subjects_dir

In [ ]:
# loading epochs
epochs = mne.read_epochs(epochs_path)

### Adding metadata from behavioural files so we have the contrast etc
Only works if no epochs were rejected, otherwise the dimensions won't match! You can check the `add_epoch_metadata.ipynb`if you want a bit more of an explanation as to what is going on. 

In [ ]:
behav_data = pd.read_csv(behav_path)
behav_data.head()

In [ ]:
# Repeat each behavioral row 4 times (as we have four triggers for each row in the behav data)
behav_data_expanded = behav_data.loc[
    behav_data.index.repeat(4)
].reset_index(drop=True)

# Add epoch type, repeating the sequence for every trial
behav_data_expanded["epoch_type"] = [
    epoch_type
    for _ in range(len(behav_data))
    for epoch_type in ["target", "mask", "response", "PAS_rating"]
]

In [ ]:
# Check the result
print(len(behav_data_expanded))
behav_data_expanded.head()

In [ ]:
epochs.metadata = behav_data_expanded

### Rejecting bad epochs
Now that we have our epochs and the metadata together, we can drop the bad epochs (if any).

In [ ]:
epochs.drop_bad(
    reject={
        "grad":4000e-13,  # unit: T / m (gradiometers)
        "mag":4e-12,      # unit: T (magnetometers)
    }
)

## Forward model
The forward model predicts how neural activity in different areas generates magnetic fields detectable by the sensors. The forward model consists of three components; the volume conductor model, the source model and a measurement model which describes the location and orientation of the sensors in relation to the head. 

### Measurement (or device) model
The measurement model describes the the positions and orientations of the MEG sensors relative to the brain anatomy, and which types of sensors are used. This information is held as metadata from our recordings. Lets plot the sensor positions in both 2d and 3d just for fun!


In [ ]:
epochs.plot_sensors(kind="3d");
epochs.plot_sensors();


### Volume conductor model
**💡 Pro tip!** Start running the code below before reading on, as it may take a little while to complete.


The volume conductor model describes the geometry and conductivity of the different tissues in the head. It models how electrical currents flow through the head, rather than where those currents originate.

The model can be derived from an MRI scan through segmentation and meshing. Segmentation identifies and classifies different tissue types, while meshing converts their geometry into a form that can be used for computational modelling.

Several types of volume conductor models exist, with different levels of complexity. The simplest models, such as spherical models, approximate the head as a sphere. They are computationally efficient but lack anatomical realism. Single-layer models provide a slightly more realistic approximation by modelling only the outer surface.

More complex models, such as boundary element models (BEM), represent the head as nested compartments—typically the skin, skull, and brain—with each compartment assumed to have uniform conductivity. This is the type of volume conductor model we will use today!

For even greater anatomical detail, finite element models (FEM) divide the head into many small elements, allowing conductivity to be specified individually. FEM can also model anisotropic conductivity, such as that of white matter, but this comes at a substantially higher computational cost.

In [ ]:
bem_model = mne.make_bem_model(subject, subjects_dir=fnames.subjects_dir)
bem_sol = mne.make_bem_solution(bem_model, verbose=None)

We can also visualise the BEM surfaces to check the head model and its alignment with the MRI.

> ⚠️ **Do not share these plots.**
> The visualisation is overlaid on the participant's MRI, which contains sensitive information. For this reason, the code below is commented out. **Make sure not to accidently share or push to GitHub.**


In [ ]:
#mne.viz.plot_bem(
#    subject, 
#    subjects_dir=subjects_dir
#);

<div class="alert alert-block alert-info">
<b> Question:</b> Which layer of the BEM does the each of the three coloured lines represent?

* Red:

* Yellow:

* Orange: 

</div>

### Source model 



In [ ]:
SRC_SPACING = "oct4" # should be set to "oct6" for real analysis, but here using oct4 for speed
src_surf = mne.setup_source_space(
    subject, 
    spacing=SRC_SPACING, 
    subjects_dir=subjects_dir
)

In [ ]:
# AGAIN COMMENTED OUT TO AVOID PUSHING TO GIT
#mne.viz.plot_bem(
#    subject=subject,
#    subjects_dir= subjects_dir,
#    src=src_surf
#);

<div class="alert alert-block alert-info">
<b> Question:</b> Which kind of tissue are the sources constrained to?
</div>

We can also create a volumetric source space.

In [ ]:
VOL_SPACING = 10.0 # mm

src_vol = mne.setup_volume_source_space(
    subject, 
    pos=VOL_SPACING, 
    subjects_dir=fnames.subjects_dir,
    bem=bem_sol # To compute a volume based source space defined with a grid of candidate dipoles inside the brain. Try removing this parameter and compare!
    )

In [ ]:
# AGAIN COMMENTED OUT TO AVOID PUSHING TO GIT
#mne.viz.plot_bem(
#    subject=subject,
#    subjects_dir=subjects_dir,
#    src=src_vol
#);

<div class="alert alert-block alert-info">
<b> Question:</b> Where are the source defined for the volumetric source space? What if you omit the bem parameter?
</div>

And a combined surface and volume source space!

In [ ]:
combined_src = src_surf + src_vol

In [ ]:
# AGAIN COMMENTED OUT TO AVOID PUSHING TO GIT
#mne.viz.plot_bem(
#    subject=subject,
#    subjects_dir=subjects_dir,
#    src=combined_src
#);

<div class="alert alert-block alert-info">
<b> Question:</b> Where are the source defined for the volumetric source space? What if you omit the bem parameter?
</div>

### Coregistration
To compute the forward model, we are combining data from the MEG system and the MR-based volume-conductor model. Therefore, it is essential to align the coordinate systems, a process known as co-registration. When dealing with MEG data, often three coordinate systems are involved:

1. **MR:** The coordinate system in which the volume-conductor model of the participants anatomy is described
2. **Device:** MEG system with sensor positions and orientations
3. **Head:** Represents the participant’s head shape, often in terms of scalp points or anatomical landmarks (e.g., nasion, left/right preauricular points).Often includes the positions of a set of coils attached to the head.


In SQUID MEG systems, the co-registration process often involves sending a weak electrical current through the coils attached to the participant's head. This current induces a magnetic field, which is detected by the MEG system's sensors. By measuring the magnetic fields generated by the coils, the location of each coil within the MEG scanners coordinate system can be calculated. This information is then used to calculate the "device-to-head" transformation, as the coil positions are known in head space, aligning the coordinate system of the MEG device with the participant's head coordinates.

This information is stored here:

In [ ]:
epochs.info["dev_head_t"]

To achieve full co-registration, an additional transformation is required. By relying on anatomical landmarks and/or scalp points, the transformation aligning the head coordinate system with the MR coordinate system can be obtained. Accurate co-registration ensures that the forward model accurately maps MEG sensor positions to the participant’s brain anatomy. Any misalignment can introduce errors into the predicted MEG signals, affecting the reliability of source localisation.

A `head_mri_t` is obtained using MNE's coregistration GUI. You can read more [here](https://mne.tools/stable/auto_tutorials/forward/20_source_alignment.html)!


In [ ]:
trans = mne.read_trans(trans_path)

Once co-registration is complete, Maxwell’s equations are applied to simulate the expected MEG signal at each sensor, as defined in the measurement model. This simulation accounts for each source specified in the source model and its propagation through the volume conductor model.

In [ ]:
fwd_surf = mne.make_forward_solution(
    epochs.info, trans=trans,
    src=src_surf, bem=bem_sol,
    mindist=1.0, n_jobs=4,
)

## Inverse model
The inverse model works backward to estimate the neural sources based on the measured magnetic fields. This problem is mathematically ill-posed (many possible solutions exist)!

In [ ]:
# create a evoked response to each of gabor patches
cond = "stimulus" # check what you called it in your own event id
stim_epochs = epochs[cond]
evoked = stim_epochs.average()

Before we can create the inverse model, we need to compute covariances between the sensors before the onset of the stimuli. This is needed for whitening the channels, i.e. normalizing the output of the magnetometers and the gradiometers such that they are comparable

In [ ]:
noise_cov = mne.compute_covariance(inst=stim_epochs, tmin=None, tmax=0)
noise_cov.plot(epochs.info)

<div class="alert alert-block alert-info">
<b>Question:</b> For the noise covariance plots, which of two kinds of sensors show the highest correlations between sensors?
</div>

## Minimum norm estimation

In [ ]:
inverse_operator = mne.minimum_norm.make_inverse_operator(
    epochs.info, 
    fwd_surf,
    noise_cov
)

In [ ]:
stc = mne.minimum_norm.apply_inverse(
    evoked, inverse_operator,
    method='MNE'
)

### Plotting source time courses
Now lets see what the source time courses look like in different areas of the brain. Freesurfer has segmented the brain into different areas using a parcellation. We can then plot the source time courses for each source in the parcels. 

In [ ]:
# read in the labels we have available from the 'aparc.a2009s' parcellation
labels = mne.read_labels_from_annot(
    subject, 
    subjects_dir=subjects_dir, 
    parc='aparc.a2009s',  
    hemi="rh" # right hemisphere. You can also change this to "lh" or "both"
)
print(labels)

In [ ]:
n_rows=7
n_cols = int(len(labels)/n_rows)
fig, axes = plt.subplots(n_rows, n_cols, sharey=True, sharex=True, figsize=(n_cols*4, n_rows*3))

for label, ax in zip(labels, axes.flatten()):
    ax.set_title(label.name)
    try:
        stc_in_label = stc.in_label(label)
    except(ValueError):
        print(f"no vertices found in {label.name}")
        continue
    ax.plot(stc_in_label.times, stc_in_label.data.T)



axes[0, 0].set_ylabel('Current density (Am)')
axes[0, 0].set_xlabel('Time (s)')

plt.tight_layout()

<div class="alert alert-block alert-info">
<b>Question:</b> Given the labels of the parcels, which areas would you expect to see activity in? Does this correspond with the plots?
</div>



### Volumetric source estimation

In [ ]:
fwd_vol = mne.make_forward_solution(
    epochs.info, trans=trans,
    src=src_vol, bem=bem_sol,
    mindist=1.0, n_jobs=4,
)

inverse_operator_vol = mne.minimum_norm.make_inverse_operator(
    epochs.info, 
    fwd_vol,
    noise_cov
)

stc_vol = mne.minimum_norm.apply_inverse(
    evoked, inverse_operator_vol,
    method='MNE'
)

In [ ]:
%matplotlib widget

# Click around the plot!
stc_vol.plot(src=src_vol, subject=subject, subjects_dir=subjects_dir, mode="glass_brain");

#### Plotting evoked contrast
Here is an example of how to take two evoked responses, apply the inverse solution to each of them and then visualise the difference in source space.

In [ ]:
epochs_pas1 = stim_epochs[stim_epochs.metadata["subjective_response"]==1]
epochs_pas4 = stim_epochs[stim_epochs.metadata["subjective_response"]==4]


# making sure that we have the same number of epochs for both conditions
rng = np.random.default_rng(42)

n = min(len(epochs_pas1), len(epochs_pas4))

epochs_pas1 = epochs_pas1[rng.choice(len(epochs_pas1), n, replace=False)]
epochs_pas4= epochs_pas4[rng.choice(len(epochs_pas4), n, replace=False)]

evoked_pas1 = epochs_pas1.average()
evoked_pas4 = epochs_pas4.average()

In [ ]:
stc_evoked_pas1 = mne.minimum_norm.apply_inverse(
    evoked_pas1, inverse_operator_vol,
    method='MNE'
)

stc_evoked_pas4 = mne.minimum_norm.apply_inverse(
    evoked_pas4, inverse_operator_vol,
    method='MNE'
)

stc_contrast = stc_evoked_pas1 - stc_evoked_pas4 # positive values = larger activation for pas 1. Negative values = larger activation for pas 4 
stc_contrast.plot(src=src_vol, subject=subject, subjects_dir=subjects_dir, mode="glass_brain");

### Morphing to `fsaverage` for group-level analysis
`fsaverage` is a **standard template brain provided by FreeSurfer**. It provides a common cortical surface onto which each participant's source estimates can be morphed.

Why do we want to morph?

Each participant has their own brain anatomy, so the same cortical vertex does not necessarily represent the same location across participants. We morph source estimates to a **common brain** so that we can compare and average data across participants.

In [ ]:
subject_to = "fsaverage"

# compute the source space for fsaverage in the same way you did for your own participant
src_vol_fsaverage = mne.setup_volume_source_space(subject_to, pos=VOL_SPACING, subjects_dir=subjects_dir)

morph = mne.compute_source_morph(
    inverse_operator_vol["src"],
    subject_from=subject,
    src_to=src_vol_fsaverage,
    subject_to=subject_to,
    subjects_dir=subjects_dir,
    verbose=False,
)

In [ ]:
# we can now morph!
stc_fsaverage = morph.apply(stc_vol)

Now we should plot on the `fsaverage` brain instead!

In [ ]:
stc_fsaverage.plot(src=src_vol_fsaverage, subject=subject_to, subjects_dir=subjects_dir, mode="glass_brain");

### Source reconstruction of epochs
So far, we have performed source reconstruction on evoked responses, which are averages across trials.
This gives us a source estimate for each trial, allowing us to preserve trial-by-trial variability and perform analyses such as multivariate analysis (more on this next week!).

In [ ]:
snr = 3

stcs_pas1 = mne.minimum_norm.apply_inverse_epochs(
    epochs_pas1, inverse_operator_vol,
    method="MNE",
    lambda2= 1.0 / 3**2
)


Play around with plotting some of the trials! Can you see some activity that makes sense at the single trial level? How variable is it?

In [ ]:
trial = 40
stcs_pas1[trial].plot(src=src_vol, subject=subject, subjects_dir=subjects_dir, mode="glass_brain");

## EXTRA: Equivalent current dipole (ECD) fit

In [ ]:
# cropping to the F100 - we don't wanna fit it all, i.e. we think dipoles are best for the early, sensory responses
evoked_100 = evoked.copy()
evoked_100.crop(0.09, 0.105)
# Fit a dipole
dip = mne.fit_dipole(
    evoked_100, 
    noise_cov, 
    bem_sol, 
    trans
)[0]

In [ ]:
colour = ["k"] * len(dip)
colour[np.argmax(dip.gof)] = "r" # the dipole with the best fit in red
dip.plot_locations(trans, subject, subjects_dir, mode="outlines", color=colour);